Scenario

A retail company receives daily product data files from multiple suppliers. Occasionally, a file fails to process due to duplicate records. Although the supplier may resend the file, the failed records are not automatically reprocessed in subsequent runs. To maintain an accurate and up-to-date product catalog, an ad hoc reprocessing is required to load the missed records, ensuring that all product additions, updates, and removals are correctly reflected in the system. This approach prevents data gaps and keeps the catalog synchronized with the latest supplier information.

CREATE catalogue and create schema and table

for target create volume and sink 

In [0]:
%sql
CREATE TABLE products
(
    id INT,
    name STRING,
    price INT,
    category STRING,
    updatedDate TIMESTAMP
);

INSERT INTO products
VALUES
(1, "iPhone", 1000, "electronics", current_timestamp()),
(2, "Macbook", 2000, "electronics", current_timestamp()),
(3, "T-Shirt", 50, "clothing", current_timestamp()),
(4, "Shirt", 100, "clothing", current_timestamp()),
(5, "Pants", 150, "clothing", current_timestamp());

In [0]:
from delta.tables import DeltaTable

try:
    dlt_obj = DeltaTable.forPath(
        spark,
        "/Volumes/pyspark_cata/source/db_volume/products_sink/"
    )

    dlt_obj.alias("trg").merge(
        df.alias("src"),
        "src.id = trg.id"
    ) \
    .whenMatchedUpdateAll(
        condition="src.updatedDate >= trg.updatedDate"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

    print("This is upserting now")

except:
    df.write.format("delta") \
        .mode("overwrite") \
        .save("/Volumes/pyspark_cata/source/db_volume/products_sink/")

In [0]:
%sql

SELECT *
FROM delta.`/Volumes/pyspark_cata/source/db_volume/products_sink/`

need to insert new row

In [0]:
%sql
CREATE TABLE products
(
    id INT,
    name STRING,
    price INT,
    category STRING,
    updatedDate TIMESTAMP
);

INSERT INTO products
VALUES
(5, "Trouser", 150, "Clothing", current_timestamp());

If you run it will fail because ..in source side we have 2 ids duplicates ..which one to take

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

df = spark.sql("select * from pyspark_cata.source.products")

# Dedup
df = df.withColumn(
    "dedup",
    row_number().over(
        Window.partitionBy('id').orderBy(desc('updatedDate'))
    )
)

df = df.filter(col('dedup') == 1).drop('dedup')

display(df)

In [0]:
from delta.tables import DeltaTable

if len(dbutils.fs.ls("/Volumes/pyspark_cata/source/db_volume/products_sink/")) > 0:

    dlt_obj = DeltaTable.forPath(
        spark,
        "/Volumes/pyspark_cata/source/db_volume/products_sink/"
    )

    dlt_obj.alias("trg").merge(
        df.alias("src"),
        "src.id = trg.id"
    ) \
    .whenMatchedUpdateAll(
        condition="src.updatedDate >= trg.updatedDate"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

    print("This is upserting now")

else:

    df.write.format("delta") \
        .mode("overwrite") \
        .save("/Volumes/pyspark_cata/source/db_volume/products_sink/")

In [0]:
%sql
SELECT * 
FROM delta.`/Volumes/pyspark_cata/source/db_volume/products_sink/`

below notes

**MERGE without condition**
df.alias("src").merge(
dlt_obj.alias("trg"),
"src.id = trg.id"
).whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()
Target table (already present)
id	name	price	updatedDate
2	Macbook	1500	2025-08-15
Backfill source data
id	name	price	updatedDate
2	Macbook	2000	2025-01-01
Step 1
Delta checks:
src.id = trg.id
Source id = 2
Target id = 2
✅ Match found
Step 2
Since a match is found, Delta executes:
.whenMatchedUpdateAll()
This means:
Update all columns from source to target.
Delta does not check whether the source record is old or new.
Step 3
Target gets overwritten.
Before
id	price	updatedDate
2	1500	2025-08-15
After
id	price	updatedDate
2	2000	2025-01-01
❌ Latest data lost.
❌ Old backfill data replaced new data.
This is the problem.
________________________________________
**MERGE WITH CONDITION**
df.alias("src").merge(
dlt_obj.alias("trg"),
"src.id = trg.id"
).whenMatchedUpdateAll(
condition="src.updatedDate >= trg.updatedDate"
).whenNotMatchedInsertAll() \
.execute()
Target table
id	name	price	updatedDate
2	Macbook	1500	2025-08-15
Backfill source
id	name	price	updatedDate
2	Macbook	2000	2025-01-01
Step 1
Delta checks:
src.id = trg.id
✅ Match found
Step 2
Before updating, Delta evaluates:
src.updatedDate >= trg.updatedDate
Substituting values:
2025-01-01 >= 2025-08-15
Result:
Plain Text
FALSE
Step 3
Because the condition is FALSE,
.whenMatchedUpdateAll()
does not run.
No update happens.
Final Result
id	name	price	updatedDate
2	Macbook	1500	2025-08-15
✅ Latest value preserved.
✅ Backfill does not overwrite current data.
________________________________________
Another Example
Target
id	price	updatedDate
2	2000	2025-01-01
Source
id	price	updatedDate
2	1500	2025-08-15
Condition check:
2025-08-15 >= 2025-01-01
✅ TRUE
Update happens.
Final Target
id	price	updatedDate
2	1500	2025-08-15
________________________________________
Interview Explanation
Without the condition, every matched record is updated. During a backfill, older records can overwrite the latest records and cause data loss.
With the condition:
condition="src.updatedDate >= trg.updatedDate"
Delta compares source and target timestamps. If the source record is newer, it updates the target. If the source record is older, Delta skips the update. This ensures that historical backfill data cannot overwrite the latest data already present in the target table.
One-line summary:
•	Without condition → matched rows always update
•	With condition → only newer records update; older backfill records are ignored